# Google Trends (pytrends) — 브라질 2016–2018 검색량 수집 (Olist 상품 기준)

Olist 데이터셋 시기(2016~2018)와 맞춰 **브라질(geo='BR')** 구글 검색 트렌드를 수집합니다.  
**Olist 실제 판매 상품 카테고리** 주문량 상위 키워드로만 수집해, 나중에 **세그먼트 변화(03-A 분기별)**와 비교할 때  
"시장 검색 관심 변화에 따라 세그먼트 구성이 어떻게 달라졌는지" 스토리로 이어갈 수 있게 합니다.  
Looker용 **CSV** 저장.

## 1. 패키지 설치 (최초 1회)

아래 셀 실행 후 커널 재시작 없이 다음 셀부터 진행하면 됩니다.

In [1]:
# pytrends 설치 (이미 있으면 스킵)
!pip install pytrends -q

## 2. 설정 및 키워드 (products.csv 전체 카테고리)

- **기간**: 2016-01-01 ~ 2018-12-31 (Olist와 동일)
- **지역**: 브라질 (`geo='BR'`)
- **키워드**: **products.csv**에 있는 **모든 제품 카테고리(distinct)**를 사용. 카테고리명의 `_`는 **띄어쓰기**로 바꿔 검색어로 씁니다. Google API 제한으로 한 번에 최대 5개씩 묶어 요청합니다.

In [2]:
import time
import pandas as pd
from pathlib import Path

try:
    from pytrends.request import TrendReq
except ImportError:
    raise ImportError("pytrends가 필요합니다. 위 셀에서 pip install pytrends 를 실행해 주세요.")

TIMEFRAME = "2016-01-01 2018-12-31"
GEO = "BR"
OUTPUT_DIR = Path.cwd()
OUTPUT_CSV = OUTPUT_DIR / "google_trends_br_2016_2018_olist.csv"

# products.csv에서 모든 제품 카테고리(distinct) 추출 → _ 를 띄어쓰기로 변환
DATA_DIR = Path.cwd() / "data"
for _ in [Path.cwd() / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (_ / "products.csv").exists():
        DATA_DIR = _
        break

if (DATA_DIR / "products.csv").exists():
    products = pd.read_csv(DATA_DIR / "products.csv")
    products["product_category_name"] = products["product_category_name"].fillna("").astype(str).str.strip()
    all_cats = products["product_category_name"].drop_duplicates()
    all_cats = all_cats[(all_cats != "") & (all_cats.str.lower() != "unknown")]
    keywords_list = [c.replace("_", " ") for c in all_cats.tolist()]
    keywords_list = [k for k in keywords_list if k]
else:
    try:
        import kagglehub
        _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
        products = pd.read_csv(_path / "olist_products_dataset.csv")
        products["product_category_name"] = products["product_category_name"].fillna("").astype(str).str.strip()
        all_cats = products["product_category_name"].drop_duplicates()
        all_cats = all_cats[(all_cats != "") & (all_cats.str.lower() != "unknown")]
        keywords_list = [c.replace("_", " ") for c in all_cats.tolist()]
        keywords_list = [k for k in keywords_list if k]
    except Exception:
        keywords_list = [
            "cama mesa banho", "beleza saude", "esporte lazer", "telefonia", "informatica acessorios",
            "automotivo", "brinquedos", "moveis decoracao", "papelaria", "utilidades domesticas",
        ]

# 한 번에 5개씩 묶음 (Google Trends 1회 요청당 최대 5개)
KEYWORDS_BATCHES = [keywords_list[i : i + 5] for i in range(0, len(keywords_list), 5)]
print("기간:", TIMEFRAME, "| 지역:", GEO)
print("키워드 출처: products.csv 전체 distinct 카테고리 (_ → 띄어쓰기)")
print("키워드 수:", len(keywords_list), "→ 묶음 수:", len(KEYWORDS_BATCHES))
print("저장 경로:", OUTPUT_CSV)
if keywords_list:
    print("키워드 목록(처음 20개):", keywords_list[:20])

기간: 2016-01-01 2018-12-31 | 지역: BR
키워드 출처: products.csv 전체 distinct 카테고리 (_ → 띄어쓰기)
키워드 수: 73 → 묶음 수: 15
저장 경로: c:\Machine_Learning\예측 대시보드 용 프로젝트\google_trends_br_2016_2018_olist.csv
키워드 목록(처음 20개): ['perfumaria', 'artes', 'esporte lazer', 'bebes', 'utilidades domesticas', 'instrumentos musicais', 'cool stuff', 'moveis decoracao', 'eletrodomesticos', 'brinquedos', 'cama mesa banho', 'construcao ferramentas seguranca', 'informatica acessorios', 'beleza saude', 'malas acessorios', 'ferramentas jardim', 'moveis escritorio', 'automotivo', 'eletronicos', 'fashion calcados']


## 3. 트렌드 수집

키워드 묶음마다 요청 후 `time.sleep`으로 rate limit을 피합니다.

In [3]:
trend_req = TrendReq(hl="pt-BR", tz=180)  # 포르투갈어(브라질), UTC-3

all_dfs = []

for i, keywords in enumerate(KEYWORDS_BATCHES):
    # 빈 문자열/None 제거
    keywords = [k for k in keywords if k]
    if not keywords:
        continue
    trend_req.build_payload(keywords, timeframe=TIMEFRAME, geo=GEO)
    df = trend_req.interest_over_time()
    if df is None or df.empty:
        print(f"묶음 {i+1}: 데이터 없음 (키워드: {keywords})")
        time.sleep(2)
        continue
    # 'isPartial' 컬럼 제거 (있을 경우)
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])
    all_dfs.append(df)
    print(f"묶음 {i+1}: {list(df.columns)} → {len(df)}행")
    time.sleep(2)  # rate limit 방지

if not all_dfs:
    raise ValueError("수집된 데이터가 없습니다. 키워드 또는 기간을 확인해 주세요.")

묶음 1: ['perfumaria', 'artes', 'esporte lazer', 'bebes', 'utilidades domesticas'] → 158행
묶음 2: ['instrumentos musicais', 'cool stuff', 'moveis decoracao', 'eletrodomesticos', 'brinquedos'] → 158행
묶음 3: ['cama mesa banho', 'construcao ferramentas seguranca', 'informatica acessorios', 'beleza saude', 'malas acessorios'] → 158행
묶음 4: ['ferramentas jardim', 'moveis escritorio', 'automotivo', 'eletronicos', 'fashion calcados'] → 158행
묶음 5: ['telefonia', 'papelaria', 'fashion bolsas e acessorios', 'pcs', 'casa construcao'] → 158행
묶음 6: ['relogios presentes', 'construcao ferramentas construcao', 'pet shop', 'eletroportateis', 'agro industria e comercio'] → 158행
묶음 7: ['moveis sala', 'sinalizacao e seguranca', 'climatizacao', 'consoles games', 'livros interesse geral'] → 158행
묶음 8: ['construcao ferramentas ferramentas', 'fashion underwear e moda praia', 'fashion roupa masculina', 'moveis cozinha area de servico jantar e jardim', 'industria comercio e negocios'] → 158행
묶음 9: ['telefonia fixa', '

## 4. Looker용 long format으로 변환 후 CSV 저장

컬럼: `date`, `keyword`, `interest` (0~100 상대 지수).  
저장 파일: **google_trends_br_2016_2018_olist.csv** — 03-A 분기별 세그먼트와 같은 기간·같은 페이지에서 "시장 검색 관심 vs 세그먼트 변화" 비교용으로 사용.

In [4]:
# wide → long: (date, keyword, interest)
long_parts = []
for df in all_dfs:
    df = df.copy().reset_index()
    # 날짜 컬럼: index가 날짜이므로 reset_index() 후 첫 번째 컬럼
    date_col = df.columns[0]
    for col in df.columns:
        if col == date_col:
            continue
        part = df[[date_col, col]].copy()
        part = part.rename(columns={col: "interest", date_col: "date"})
        part["keyword"] = col
        long_parts.append(part)

trend_long = pd.concat(long_parts, ignore_index=True)
trend_long = trend_long[["date", "keyword", "interest"]]

# date를 문자열로 통일 (Looker/Sheet 호환)
trend_long["date"] = pd.to_datetime(trend_long["date"]).dt.strftime("%Y-%m-%d")

trend_long.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("저장 완료:", OUTPUT_CSV)
print("행 수:", len(trend_long))
trend_long.head(10)

저장 완료: c:\Machine_Learning\예측 대시보드 용 프로젝트\google_trends_br_2016_2018_olist.csv
행 수: 11534


,date,keyword,interest
0,2015-12-27,perfumaria,9
1,2016-01-03,perfumaria,9
2,2016-01-10,perfumaria,10
3,2016-01-17,perfumaria,11
4,2016-01-24,perfumaria,10
5,2016-01-31,perfumaria,10
6,2016-02-07,perfumaria,9
7,2016-02-14,perfumaria,10
8,2016-02-21,perfumaria,9
9,2016-02-28,perfumaria,10


## 5. (참고) 커스텀 기간이 429 에러를 줄 때

Google에서 `429 Too Many Requests`가 나오면, 아래처럼 **최근 5년** 데이터를 받은 뒤 2016–2018만 잘라서 사용할 수 있습니다.  
지금 시점에서 'today 5-y'는 2020–2025 구간이므로 **2016–2018은 커스텀 기간이 필수**입니다.  
한 번에 키워드 수를 줄이거나, 요청 간 대기 시간을 3~5초로 늘려 보세요.